In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import os
import time

In [ ]:
# Constants
RANDOMSTATE = 1082
TRAIN_SIZE = 0.8
DATA_PATH = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
MODEL_NAME = "Logistic Regression"
DATASET_NAME = "spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../../UAVATTACK_Results.csv"

In [ ]:
# Load the dataset
try:
    df = pd.read_csv("../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv' was not found.")
    exit()

In [ ]:
# Data Preprocessing

# Clean up column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Drop label
drop_cols = ['timestamp', 'label']
X = df.drop(columns=drop_cols)
y = df['label']

if y.dtype == object or y.dtype.name == 'category':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=TRAIN_SIZE, random_state=RANDOMSTATE, stratify=y
)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train the model
model = LogisticRegression(
    random_state=RANDOMSTATE,
    max_iter=2000,
    solver='saga'
)
start_time = time.time()
model.fit(X_train_scaled, y_train)
train_time = time.time() - start_time

In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference


_proba = model.predict_proba(X_test_scaled) if hasattr(model, "predict_proba") else None
_timing = time_inference(model.predict, X_test_scaled)

row = evaluate(
    y_test,
    y_pred,
    model_name=MODEL_NAME,
    dataset_name=DATASET_NAME,
    family="Regular",
    labels=[0, 1],
    class_names=["benign", "malicious"],
    y_proba=_proba,
    y_train=y_train,
    train_time_sec=train_time,
    num_epochs="N/A",
    parameters=model.get_params(),
    seed=RANDOMSTATE,
    n_features=X_test_scaled.shape[1],
    n_train=len(y_train),
    timing=_timing,
)
append_row(row, RESULTS_CSV)
print(summary(row))
print(f"Results appended to {RESULTS_CSV}")